# Exercise 2

In [ ]:
from functions import RandomnessTests
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(42)
import time
import tracemalloc

In [ ]:
from importlib import reload
import functions

reload(functions)

from functions import RandomnessTests

## Part 1

In [ ]:
p1, p2, p3 = 0.05, 0.3, 0.7
X005 = np.random.geometric(p1, size=10000)
X03 = np.random.geometric(p2, size=10000)
X07 = np.random.geometric(p3, size=10000)

results = RandomnessTests.compare_datasets(
    [X005, X03, X07],
    labels=["geo p=0.05", "geo p=0.3", "geo p=0.7"],
    correlation_lags=[1, 2, 5, 10]
)

## Part 2

### direct (crude) method
Mest grundlæggende måde at generere en diskret stokastisk variabel på. 
Ideen er at man bruger et uniformt tal og finder hvilket interval U falder i, baseret på den kumulative fordelingsfunktion.

In [ ]:
xi = [1, 2, 3, 4, 5, 6]
PXxi = [7/48, 5/48, 1/8, 1/16, 1/4, 5/16]

In [ ]:
def direct(x, p, n_samples):
    u = random.random()
    P = []
    current = 0
    for pi in p:
        current += pi
        P.append(current)

    samples = []
    for _ in range(n_samples):
        u = random.random()  # U ~ Uniform(0,1)

        # Linear search for at finde det første P[i], der er større end eller lig med u
        for i in range(len(P)):
            if u <= P[i]:
                samples.append(x[i])
                break

    return samples

In [ ]:
direct_samples = direct(xi, PXxi, 10000)

### rejection method
Alternativ måde til at generere en diskret stokastisk variabel på. 
Ideen er at man foreslår et udfald og accepterer det kun med en vis sansynlighed. Hvis det afvises, prøver man igen.

In [ ]:
def simple_rejection(x, p, n_samples):
    c = max(p)
    samples = []
    k = len(x)

    while len(samples) < n_samples:
        I = int(np.floor(k*random.random())) # nul indeksering i python så inge plus en

        if random.random() <= p[I]/c: 
            samples.append(x[I])
    return samples

In [ ]:
simple_rejection_samples = simple_rejection(xi, PXxi, 10000)

### alias method
Alias metoden er en hurtig måde at generere fra en vilkårlig diskret fordeling.
Ideen er at man omdanner en vilkårlig diskret fordeling til k "buckets" hver med en primlr værdi og en alias-værdi. Under samplingen kræver det så kun et opslag og en sammenligning for at finde det korrekte udfald.

In [ ]:
def alias_method(p, x, n_samples):

    k = len(p)

    # Konstruer lookup tabeller
    q = [pi * k for pi in p]
    prob = [0] * k
    alias = [0] * k

    small = []
    large = []

    for i in range(k):
        if q[i] < 1:
            small.append(i)
        else:
            large.append(i)

    # Fyld buckets
    while small and large:
        s = small.pop()
        l = large.pop()

        prob[s] = q[s]
        alias[s] = l

        q[l] = q[l] - (1 - q[s])

        if q[l] < 1:
            small.append(l)
        else:
            large.append(l)

    # Resten sættes til 1
    for i in small + large:
        prob[i] = 1
        alias[i] = i

    # --- 2. Sampling ---
    samples = []
    for _ in range(n_samples):
        i = int(random.random() * k)  # bucket index
        if random.random() < prob[i]:
            samples.append(x[i])
        else:
            samples.append(x[alias[i]])

    return samples


In [ ]:
alias_samples = alias_method(PXxi, xi, 10000)

## Part 3

### Ease of implementation
Hvor let er det at implementere?

### Computational efficency
Hvor lang tid tager det?

In [ ]:
def measure_time(func, *args, repeats=10, **kwargs):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        func(*args, **kwargs)
        end = time.perf_counter()
        times.append(end - start)

    return {
        "mean_time": np.mean(times),
        "std_time": np.std(times)
    }

In [ ]:
print(f'direct_method: {measure_time(direct, xi, PXxi, 10000)["mean_time"]:.4f} +- {measure_time(direct, xi, PXxi, 10000)["std_time"]:.4f} seconds')
print(f'rejection_method: {measure_time(simple_rejection, xi, PXxi, 10000)["mean_time"]:.4f} +- {measure_time(simple_rejection, xi, PXxi, 10000)["std_time"]:.4f} seconds')
print(f'alias_method: {measure_time(alias_method, PXxi, xi, 10000)["mean_time"]:.4f} +- {measure_time(alias_method, PXxi, xi, 10000)["std_time"]:.4f} seconds')

### Memory requirements
Hvor meget hukommelse kræver det?

In [ ]:
def measure_memory(func, *args, **kwargs):
    tracemalloc.start()

    result = func(*args, **kwargs)

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "current_memory": current,
        "peak_memory": peak,
        "result": result
    }

In [ ]:
print(f'direct_method memory: {measure_memory(direct, xi, PXxi, 10000)["peak_memory"] / 1024:.2f} KB')
print(f'rejection_method memory: {measure_memory(simple_rejection, xi, PXxi, 10000)["peak_memory"] / 1024:.2f} KB')
print(f'alias_method memory: {measure_memory(alias_method, PXxi, xi, 10000)["peak_memory"] / 1024:.2f} KB')

### Accuracy of the generated distributions
Hvor godt matcher de genererede fordelinger den ønskede fordeling?

In [ ]:
def measure_accuracy(data):
    results = RandomnessTests.analyze(
        data,
        histogram=False,
        scatter=False,
        chi_square=True,
        ks=True,
        wald_wolf=True,
        knuth=True,
        up_down=True,
        correlation=False
    )

    return results

In [ ]:
print(f'direct method accuracy: {measure_accuracy(direct_samples)}')
print(f'rejection method accuracy: {measure_accuracy(simple_rejection_samples)}')
print(f'alias method accuracy: {measure_accuracy(alias_samples)}')

## Sammenligning

In [ ]:
def evaluate_generator(name, func, *args, **kwargs):

    print(f"\n===== Evaluating {name} =====")

    # Time
    t = measure_time(func, *args, **kwargs)
    print("Time:", t)

    # Memory
    m = measure_memory(func, *args, **kwargs)
    data = m["result"]
    print("Peak memory:", m["peak_memory"])

    # Accuracy
    acc = measure_accuracy(data)
    print("Accuracy tests done")

    return {
        "time": t,
        "memory": m,
        "accuracy": acc
    }

In [ ]:
gens = [
    ("Direct Method", direct, {"x": xi, "p": PXxi, "n_samples": 10000}),
    ("Rejection Method", simple_rejection, {"x": xi, "p": PXxi, "n_samples": 10000}),
    ("Alias Method", alias_method, {"p": PXxi, "x": xi, "n_samples": 10000}),
]

all_results = {}

for name, func, kwargs in gens:
    all_results[name] = evaluate_generator(
        name,
        func,
        **kwargs
    )

In [ ]:
for name, func, kwargs in gens:
    result = evaluate_generator(name, func, **kwargs)

    print("\n", name)
    print(result.keys())
    print(result.get("accuracy"))

    all_results[name] = result

### Plots

In [ ]:
def z_normalize(values):
    # Positivt = over gennemsnittet, Negativt = under gennemsnittet
    values = np.array(values, dtype=float)
    if np.std(values) == 0:
        return values
    return (values - np.mean(values)) / np.std(values)

def plot_all_results(all_results, normalize=True):
    names = list(all_results.keys())

    # time
    times = [
        all_results[name]["time"]["mean_time"]
        for name in names
    ]

    plt.figure()
    plt.bar(names, z_normalize(times) if normalize else times)
    plt.title("Computational Efficiency (Time)")
    plt.ylabel("Seconds")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # memory
    memory = [
        all_results[name]["memory"]["peak_memory"] / 1024
        for name in names
    ]

    plt.figure()
    plt.bar(names, z_normalize(memory) if normalize else memory)
    plt.title("Memory Usage (Peak)")
    plt.ylabel("KB")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # accuracy
    ks_values = []

    for name in names:
        ks = all_results[name]["accuracy"]["ks"][0]
        ks_values.append(ks)

    plt.figure()
    plt.bar(names, z_normalize(ks_values) if normalize else ks_values)
    plt.title("Distribution Accuracy (KS statistic)")
    plt.ylabel("KS Dn")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    chi2_values = []

    for name in names:
        chi2 = all_results[name]["accuracy"]["chi_square"][0]
        chi2_values.append(chi2)

    plt.figure()
    plt.bar(names, z_normalize(chi2_values) if normalize else chi2_values)
    plt.title("Distribution Accuracy (Chi-Square)")
    plt.ylabel("Chi-Square Statistic")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_all_results(all_results)

In [ ]:
plot_all_results(all_results, normalize=False)

## Part 4